In [ ]:
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')

In [ ]:
import numpy as np
import cv2
import numpy as np
import matplotlib.pyplot as plt
from wisardpkg import ClusWisard
from wisard_object_tracker.src.utils import tracker_utils

In [ ]:
def generate_search_regions_spiral(prev_bbox, frame_shape, search_region_scale=1.5, step_size=5):
    """
    Gera regiões de busca começando do bbox central, percorrendo círculos 
    step a step até atingir o raio máximo definido por search_region_scale.
    """
    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2
    raio_max = (max(w, h) / 2) * search_region_scale

    search_bboxes = []

    # Começa incluindo o bbox original
    search_bboxes.append((x, y, w, h))

    # Raio inicial
    raio = step_size
    while raio <= raio_max:
        circunferencia = 2 * np.pi * raio
        num_steps = max(8, int(circunferencia // step_size))
        for i in range(num_steps):
            theta = 2 * np.pi * i / num_steps
            px = int(center_x + raio * np.cos(theta) - w // 2)
            py = int(center_y + raio * np.sin(theta) - h // 2)
            px = max(0, min(frame_shape[1] - w, px))
            py = max(0, min(frame_shape[0] - h, py))
            search_bboxes.append((px, py, w, h))
        raio += step_size

    return search_bboxes

def show_patch(patch):
    plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()



# Caminho da imagem
img_path = "/mnt/c/Users/Isabella/tcc/wisard_object_tracker/data/tiger2/imgs/img00000.png"
img = cv2.imread(img_path)

# Primeiro bounding box
prev_bbox = (16, 30, 34, 39)  # x, y, w, h
search_regions = generate_search_regions_spiral(prev_bbox, img.shape[:2], step_size=1)

In [ ]:
patch_1 = tracker_utils.extract_patch(img, prev_bbox)
pattern = tracker_utils.preprocess_frame(patch_1, target_size=(32, 32))
dataset1 = [pattern.tolist()]


In [ ]:
clus = ClusWisard(
            2, # address size
            0.3, # min score
            20, # threshold
            20, # discriminator limit
            bleachingActivated=True,
            returnActivationDegree=True,
            returnConfidence=True,
            returnClassesDegrees=True
        )
clus.train(dataset1, ["tiger"])

In [ ]:
clus.json()

In [ ]:
for i in range(0, 2501, 100):  # vai de 0 até 2500, pulando de 300 em 300
    patch = tracker_utils.extract_patch(img, search_regions[i])
    pattern = tracker_utils.preprocess_frame(patch, target_size=(32, 32))
    dataset = [pattern.tolist()]
    # clus.train(dataset, ["tiger"])
    # Mostra o patch
    show_patch(patch)

    # Classifica e printa o resultado
    result = clus.classify(dataset)
    print(f"Região {i} -> Classificação: {result}")


In [ ]:
patch = tracker_utils.extract_patch(img, search_regions[700])
pattern = tracker_utils.preprocess_frame(patch, target_size=(32, 32))
dataset = [pattern.tolist()]
show_patch(patch)
clus.classify(dataset)

In [ ]:
clus.train(dataset, ["tiger"])

In [ ]:
import json

# pega a string do ClusWisard
json_str = clus.json()

# transforma em dict
clus_dict = json.loads(json_str)

clusters = clus_dict["clusters"]

# Número de clusters (classes)
num_clusters = len(clusters)

# Número de discriminadores por cluster
discriminadores_por_cluster = {nome: len(c["discriminators"]) for nome, c in clusters.items()}

print("Número de clusters:", num_clusters)
print("Discriminadores por cluster:", discriminadores_por_cluster)


In [ ]:
patterns = clus.getMentalImages()

key_iterator = iter(patterns)
key = next(key_iterator)
cluster = patterns[key]
img_vector=cluster[3]
side = int(np.sqrt(len(img_vector)))  # tenta descobrir tamanho quadrado
img_matrix = np.array(img_vector).reshape(side, side)

# plota
plt.imshow(img_matrix, cmap="gray")  # "gray" deixa 0=preto, 1=branco
plt.axis("off")
plt.show()